# BITS Pilani — MLOps (AIMLCZG523) Assignment 01
## End-to-End ML Pipeline: Heart Disease Risk Prediction
---
**Objective:** Build, validate, track with MLflow, containerize, and deploy a production-grade machine learning classifier for predicting clinical heart disease risk.

### 1. Ingestion & Environment Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import joblib
import mlflow
import mlflow.sklearn

sns.set_theme(style='whitegrid')

### 2. Dataset Loading & Inspection

In [ ]:
data_path = '../data/heart_disease.csv'
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape}")
display(df.head())
print("\nMissing values per feature:")
print(df.isnull().sum())

### 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class Balance
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Class Balance (0: Healthy, 1: Heart Disease)')
plt.xlabel('Target Outcome')
plt.ylabel('Patient Count')
plt.show()

# Correlation Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Pearson Correlation Heatmap')
plt.show()

### 4. Feature Engineering & Preprocessing Pipeline

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

### 5. Model Development & MLflow Experiment Tracking

In [ ]:
mlflow.set_experiment('Heart_Disease_Notebook_Experiment')

models = {
    'Logistic_Regression': (LogisticRegression(max_iter=1000, random_state=42), {'classifier__C': [0.1, 1.0, 10.0]}),
    'Random_Forest': (RandomForestClassifier(random_state=42), {'classifier__n_estimators': [50, 100], 'classifier__max_depth': [3, 5, 8]})
}

best_score = 0
best_pipeline = None

for name, (clf, param_grid) in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    with mlflow.start_run(run_name=name):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
        grid.fit(X_train, y_train)
        
        y_pred = grid.predict(X_test)
        y_prob = grid.predict_proba(X_test)[:, 1]
        
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        
        mlflow.log_params(grid.best_params_)
        mlflow.log_metric('test_accuracy', acc)
        mlflow.log_metric('test_roc_auc', auc)
        mlflow.sklearn.log_model(grid.best_estimator_, 'model')
        
        print(f"{name} -> Best CV Score: {grid.best_score_:.4f} | Test Acc: {acc:.4f} | Test AUC: {auc:.4f}")
        
        if auc > best_score:
            best_score = auc
            best_pipeline = grid.best_estimator_

os.makedirs('../model', exist_ok=True)
joblib.dump(best_pipeline, '../model/heart_disease_pipeline.joblib')
print("\n[+] Production model successfully serialized to ../model/heart_disease_pipeline.joblib")